# Stiffness 10 — conditioning

Every other notebook in this chapter asks *how much does this cost?* This one asks a different question, because its answer is not a number of steps.

A steric mass action column: salt plus one protein, loaded at low salt, washed, then eluted with a salt gradient. The binding flux is a **difference of two terms**

$$k_a\,c_p\,(\Lambda - (\nu+\sigma)q)^{\nu} \;-\; k_d\,q\,c_{salt}^{\nu},$$

and both carry a quantity of order hundreds raised to the power $\nu$. At $\nu = 12$ and a salt of 300 mol/m³, the second factor alone is about $5\cdot10^{29}$. Those two numbers have to cancel to whatever precision the solution needs.

| | |
|---|---|
| column | 0.1 m, 150 cm/h, LRMP, 50 cells |
| ionic capacity $\Lambda$ | 1200 mol/m³ |
| gradient | 50 → 500 mol/m³ salt |
| protein | 0.05 mol/m³, elutes around 340 mol/m³ |

Two knobs, **neither of which changes the chemistry**. The characteristic charge $\nu$ sets how far the magnitudes are blown up, with the equilibrium constant retuned alongside it so the protein keeps eluting at the same salt. The kinetics factor scales $k_a$ and $k_d$ together, which leaves the equilibrium — and therefore the chromatogram — exactly where it was.

The first notebook of this chapter established that scaling binding kinetics is free. So anything that happens here is not stiffness. It is arithmetic.

In [ ]:
# This notebook lives one folder below the workshop root, while `utility`
# and everything under `data/` are addressed relative to that root.
# Step up into it before importing anything.
import os
import sys

root = os.path.abspath(os.curdir)
while not os.path.isdir(os.path.join(root, 'utility')):
    root, previous = os.path.dirname(root), root
    if root == previous:
        raise RuntimeError('no folder containing utility/ above this notebook')

os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)

In [ ]:
import utility.conditioning_explorer as explorer

explorer.show_setting()

## What to watch

The column to watch is `k_d c_salt^nu` — the size of the number being cancelled — and the one next to it, the **protein peak**, which tells you whether anything has changed about the answer.

Unlike every other study here, some settings will simply **fail**. The record keeps them, with the reason.

In [ ]:
from ipywidgets import interact

show_conditioning = explorer.make_explorer()

interact(show_conditioning, **show_conditioning.sliders);

## What happened

**It works perfectly, identically, and then it stops working.**

| $\nu$ | largest factor that runs | $k_d c_{salt}^{\nu}$ there | next factor |
|---|---|---|---|
| 4 | 10¹⁴ | 8.1·10²³ | still fine |
| 8 | 10⁸ | 6.6·10²⁷ | **IDA_ERR_FAIL** |
| 12 | 10² | 5.3·10³¹ | **IDA_ERR_FAIL** |

Up to the last working row the chromatogram is identical to **0.0000 %**, the step count is flat, and the error test failures do not move. There is no warm-up, no slowdown, no degradation of the answer. The simulation is exactly as good as it was — until it is not a simulation at all.

**And the failure arrives at a section boundary**, `t = 240 s`, where the gradient starts. That is where the solver has to re-initialise, so it is where ill-conditioned arithmetic is asked the hardest question first.

**This is not stiffness.** Scaling $k_a$ and $k_d$ together is the very knob that Stiffness 1 swept over fourteen decades for free. The chemistry here is unchanged, the time scales are irrelevant, and the integrator is not working harder. What changed is that two numbers of order $10^{29}$ have to cancel in double precision, which carries about sixteen digits.

<details>
<summary><b>Click to learn more: this is a real support case</b></summary>

The `Fail case 908` notebook in this chapter is exactly this failure, found in the wild. A user translated a kinetic constant from another simulator, obtaining $k_d = 5.6\cdot10^{7}$ with $\nu \approx 10$, and the simulation died at the first step with `IDA_ERR_FAIL`.

Three of the four settings one would reach for — `max_errtest_fail`, `max_newton_iter`, tightening `abstol` — do nothing, because none of them addresses arithmetic. What fixed it was recognising that the rate constants were not physical: at that $\nu$, nothing in the model is kinetic, and the honest configuration is rapid equilibrium, where only the **ratio** $k_a/k_d$ enters and the magnitudes vanish.

</details>

> **Take-away.** Look at what your rate constants get multiplied by, not just at the constants. In SMA the effective terms are $k_a \bar{q}^{\nu}$ and $k_d c_{salt}^{\nu}$, and with $\nu \approx 10$ those exponentials dominate completely. A model can be perfectly well posed, perfectly fast, and arithmetically impossible — and it will tell you by failing outright rather than by slowing down. If a simulation dies at a section boundary with a solver error, check the magnitudes in your isotherm before you touch a tolerance.